In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TABLE workspace.silver.trips AS
WITH src AS (
    SELECT
        *
      , year    AS file_year
      , month   AS file_month
    FROM
        workspace.bronze.trips
)
SELECT
    VendorID                    AS vendor_id
  , tpep_pickup_datetime        AS pickup_at
  , tpep_dropoff_datetime       AS dropoff_at
  , CASE WHEN passenger_count = 0 THEN NULL ELSE CAST(passenger_count AS INT) END AS passenger_count
  , CAST(RatecodeID AS INT)     AS rate_code_id
  , trip_distance
  , store_and_fwd_flag
  , PULocationID                AS pickup_location_id
  , DOLocationID                AS dropoff_location_id
  , CAST(payment_type AS INT)   AS payment_type
  , fare_amount
  , extra
  , mta_tax
  , tip_amount
  , tolls_amount
  , improvement_surcharge
  , congestion_surcharge
  , airport_fee
  , total_amount
  , fare_amount < 0             AS is_negative_fare
  , trip_distance = 0           AS is_zero_distance
  , file_year
  , file_month
  , _source_file
  , _ingested_at
FROM
    src
WHERE
    tpep_dropoff_datetime >= tpep_pickup_datetime
AND year(tpep_pickup_datetime)  = file_year
AND month(tpep_pickup_datetime) = file_month;

In [0]:
SELECT
    COALESCE(b.year, s.file_year)               AS year
  , COALESCE(b.month, s.file_month)             AS month
  , b.bronze_rows
  , s.silver_rows
  , b.bronze_rows - COALESCE(s.silver_rows, 0)  AS dropped
FROM
    (SELECT year, month, COUNT(*) AS bronze_rows FROM workspace.bronze.trips GROUP BY year, month) b
FULL OUTER JOIN
    (SELECT file_year, file_month, COUNT(*) AS silver_rows FROM workspace.silver.trips GROUP BY file_year, file_month) s
    ON  b.year  = s.file_year
    AND b.month = s.file_month
ORDER BY
    year, month;